In [ ]:
import pyspark.sql.functions as F

In [ ]:
dbutils.widgets.text("filename", "")
filename = dbutils.widgets.get("filename")

In [ ]:
source_path = f"/Volumes/tihim_project/landing/source_files/{filename}" 
schema_path     = f"/Volumes/tihim_project/ops/stream_state/schemas/bronze/{filename}"
checkpoint_path = f"/Volumes/tihim_project/ops/stream_state/checkpoints/bronze/{filename}"

In [ ]:
df_stream = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", schema_path) \
    .load(source_path) \
    .withColumn("ingestion_date", F.current_timestamp()) \
    .withColumn("source_file", F.col("_metadata.file_path"))

In [ ]:
query = df_stream.writeStream \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .toTable(f"tihim_project.bronze.{filename}")


query.awaitTermination()